## Import

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import matplotlib.pyplot as plt


device = 'cuda:2'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Data generation

In [4]:
from datasets import Spiral

n, d = 10000, 64                                
true_rho = 0.7                                   # < higher rho, higher MI
v = 3.14/2

dataset = Spiral(rho=true_rho, dim=d, v=v)
X, Y = dataset.sample(n=10000)
X, Y = X.to(device).clone().detach(), Y.to(device).clone().detach()


print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", dataset.MI())

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 10.773512852220248


## MI estimation

In [5]:
class Hyperparams(object):
    def __init__(self): 
        self.lr = 5e-4
        self.bs = 500
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [4]:
## Mutual information neural diffusion estimate (MINDE)
from estimators import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500


finished: t= 0 loss= 1.4955682754516602 loss val= 1.508802890777588 best val loss= 1.508802890777588 best t= 0


finished: t= 63 loss= 0.7335732579231262 loss val= 0.7673870325088501 best val loss= 0.6941083669662476 best t= 53


finished: t= 126 loss= 0.7612102627754211 loss val= 0.7416495084762573 best val loss= 0.6941083669662476 best t= 53


finished: t= 189 loss= 0.7640083432197571 loss val= 0.7277002334594727 best val loss= 0.6941083669662476 best t= 53


...

finished: t= 693 loss= 0.646149218082428 loss val= 0.7320349216461182 best val loss= 0.6809502243995667 best t= 363


finished: t= 756 loss= 0.7338274121284485 loss val= 0.7565888166427612 best val loss= 0.6809502243995667 best t= 363


finished: t= 819 loss= 0.773039698600769 loss val= 0.7458082437515259 best val loss= 0.6809502243995667 best t= 363




true MI: 10.773512852220248


est MI: 10.222756118774415


In [5]:
## Mutual information neural estimate (MINE)
from estimators import MINE

estimator = MINE(architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.017472945153713226 loss val= 0.017278587445616722 best val loss= 0.017278587445616722 best t= 0


finished: t= 63 loss= -6.199862480163574 loss val= -5.825044631958008 best val loss= -5.98870849609375 best t= 60


finished: t= 126 loss= -5.717525959014893 loss val= -5.809330463409424 best val loss= -6.1706695556640625 best t= 120


finished: t= 189 loss= -5.817143440246582 loss val= -6.093094825744629 best val loss= -6.2081708908081055 best t= 160


finished: t= 252 loss= -6.001667022705078 loss val= -5.916452407836914 best val loss= -6.238798141479492 best t= 197


...

finished: t= 567 loss= -5.678356647491455 loss val= -5.853240013122559 best val loss= -6.316197395324707 best t= 543


finished: t= 630 loss= -5.305305480957031 loss val= -6.033124923706055 best val loss= -6.316197395324707 best t= 543


finished: t= 693 loss= -5.380235195159912 loss val= -5.603885650634766 best val loss= -6.316197395324707 best t= 543




true MI: 10.773512852220248


est MI: 7.588613510131836


In [6]:
## Vector copula-based MI estimation
from estimators import VCE

estimator = VCE(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 2.046576976776123 loss val= 1.9988155364990234 best val loss= 1.9988155364990234 best t= 0
finished: t= 126 loss= 1.5253374576568604 loss val= 1.6337034702301025 best val loss= 1.603434681892395 best t= 97
finished: t= 252 loss= 1.5650382041931152 loss val= 1.6344995498657227 best val loss= 1.603434681892395 best t= 97


finished: t= 0 loss= 2.021775007247925 loss val= 1.9940017461776733 best val loss= 1.9940017461776733 best t= 0
finished: t= 126 loss= 1.5461424589157104 loss val= 1.6186866760253906 best val loss= 1.6059472560882568 best t= 68
finished: t= 252 loss= 1.5946882963180542 loss val= 1.6430518627166748 best val loss= 1.5988456010818481 best t= 175


finished: t= 0 loss= 573.6554565429688 loss val= 590.16357421875 best val loss= 590.16357421875 best t= 0
finished: t= 251 loss= 78.30937194824219 loss val= 83.37836456298828 best val loss= 83.37836456298828 best t= 251
finished: t= 502 loss= 77.66556549072266 loss val= 83.3385009765625 best val loss= 83.338

In [5]:
## MI estimate via flows (MIENF)
from estimators import MIENF

estimator = MIENF(hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.MI())
print('est MI:', estimator.MI(X, Y))

MIENF (K=1), joint learning True 

finished: t= 0 loss= 9451.5185546875 loss val= 9312.685546875 best val loss= 9312.685546875 best t= 0
finished: t= 101 loss= 86.12310028076172 loss val= 87.20240783691406 best val loss= 87.20240783691406 best t= 101
finished: t= 202 loss= 79.40116882324219 loss val= 83.94515991210938 best val loss= 83.46994018554688 best t= 173
finished: t= 303 loss= 77.64783477783203 loss val= 87.6287612915039 best val loss= 83.46994018554688 best t= 173


true MI: 10.773512852220248
est MI: 10.18903923034668
